## Data recalibration
 This project has been going on for a while,and we have attempted to make the deadline for  two conferences:

 First ICWSM in september, then web conference in october ther the web for good track for the same conference and finally ICWSm deadline in January

 Now we are looking at the ICWSM conference for 2027. The first deadline is in May 2026.


## The goal of this script:
To consolidate all the collected data, the reviews that have been classified, those that have not and what next

## libraries to import

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import BallTree
from tqdm import tqdm
from fuzzywuzzy import fuzz
import warnings
import os
warnings.filterwarnings('ignore')

In [ ]:
#pip install fuzzywuzzy

## The Groundtruth data from baderleben

In [ ]:
baederleben_pools = pd.read_csv("//home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/baederleben_pools_07042026.csv")
baederleben_pools.info()

In [ ]:
# drop duplicates on the longitude and latitude columns
## show duplicates
duplicates = baederleben_pools[baederleben_pools.duplicated(subset=["longitude", "latitude","postalcode","city"], keep=False)]
duplicates.info()

In [ ]:
# merge duplicates so we have the alternatives for the columns with different values
baederleben_pools = baederleben_pools.groupby(["longitude", "latitude", "postalcode", "city"], as_index=False).agg({
    "pool_id": "first",
    "pool_type": lambda x: ", ".join(x.dropna().unique()),
    "pool_name": lambda x: ", ".join(x.dropna().unique()),
    "municipal_code": "first",
    "street": lambda x: ", ".join(x.dropna().unique()),
    "pool_website": lambda x: ", ".join(x.dropna().unique())
})
# baederleben_pools = baederleben_pools.drop_duplicates(subset=["longitude", "latitude"])
#baederleben_pools.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/baederleben_pools_to_use_in_project_17042026.csv", index=False)
baederleben_pools.info()

In [ ]:
# correct andconvert coordinates to float
def correct_coordinates_in_place(df, lat_col='latitude', lon_col='longitude'):
    """
    Correct the coordinate columns in place by replacing first comma with dot and removing others
    """
    # Store original counts for reporting
    original_lat_problems = df[lat_col].astype(str).str.contains(',').sum()
    original_lon_problems = df[lon_col].astype(str).str.contains(',').sum()
    
    # Function to apply the transformation
    def transform_coordinate(value):
        if isinstance(value, str) and ',' in value:
            parts = value.split(',')
            return parts[0] + '.' + ''.join(parts[1:])
        return value
    
    # Apply transformations
    df[lat_col] = df[lat_col].apply(transform_coordinate)
    df[lon_col] = df[lon_col].apply(transform_coordinate)
    
    # Convert to float64
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # Report what was fixed
    print(f"Fixed {original_lat_problems} latitude values and {original_lon_problems} longitude values")
    print(f"Data types after correction:")
    print(f"  {lat_col}: {df[lat_col].dtype}")
    print(f"  {lon_col}: {df[lon_col].dtype}")
    
    return df

In [ ]:
# Apply the correction to to tjh
print("Before correction:")
print(f"baederleben_pools latitude dtype: {baederleben_pools['latitude'].dtype}")
print(f"baederleben_pools longitude dtype: {baederleben_pools['longitude'].dtype}")

# Correct the dataframe IN PLACE
baederleben_pools = correct_coordinates_in_place(baederleben_pools)

print("\nAfter correction:")
print(f"Sample latitude values: {baederleben_pools['latitude'].head()}")
print(f"Sample longitude values: {baederleben_pools['longitude'].head()}")

In [ ]:
baederleben_pools.info()

## Collected Google Maps POIs

In [ ]:
#!pip install -q fastparquet

In [ ]:
#all collected google pois
google_pois = pd.read_parquet(
    "/home/ethel/Desktop/swimming_pool_project/clean_datasets/pois_within_shapefile.parquet",
    engine="fastparquet",
)
google_pois.info()

In [ ]:
google_pois[google_pois['place_id'].isin(missing_uids['place_id'])][['place_id', 'place_name', 'latitude', 'longitude']].drop_duplicates()

## lets select the  swimming pools of interest

In [ ]:

list_place_types = ['aquatic center', 'indoor swimming pool', 'lake', 'lake shore swimming area','lido',
                    'outdoor bath', 'outdoor swimming pool', 'public bath', 'public bathroom', 
                    'public sauna', 'public beach','public swimming pool', 'recreation center', 'spa','spa and health club',
                    'sports complex', 'swimming facility', 'swimming lake','swimming pool', 'thermal baths',
                    'water', 'water park']

In [ ]:
#first  make the promary place_type lower case
google_pois["primary_place_type"] = google_pois["primary_place_type"].str.lower()
google_pools= google_pois[google_pois["primary_place_type"].isin(list_place_types)]
google_pools.info()

In [ ]:
### check the primamry place type of the google pools 
google_pois[~google_pois["unique_identifier"].isin(google_pools["unique_identifier"])]["primary_place_type"].unique()

## since we already checked this before let's crosscheck for those that we already matched or merged

In [ ]:
#final pool list from the 24.11.2025
matched_pools = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/swimmingpool_list_to_use_25112025.csv")
#drop columns 23 to 38 and 43 to 55
columns_to_drop = list(matched_pools.columns[23:39]) + list(matched_pools.columns[43:56])
matched_pools = matched_pools.drop(columns=columns_to_drop)
matched_pools.info()

In [ ]:
##the 848 pools we missed int he old admin list
pools_848 = pd.read_parquet("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/general_poi_info.parquet"
                            , engine="fastparquet")
#drop null  columns
pools_848 = pools_848.dropna(axis=1, how='all')
pools_848.info()

#the matched 848 pools
pools_848_matched = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/848_pools_gmaps_12022026_with_uids.csv")
pools_848_matched.info()

In [ ]:
## merged data on the 848 pools
pools_848_merged = pd.merge(pools_848, pools_848_matched, on="unique_identifier", how="left")
#drop all null columns
pools_848_merged = pools_848_merged.dropna(axis=1, how="all")

pools_848_merged.info()

In [ ]:
## now merge the 848 pools with the matched data
all_matched_pools=pd.concat([matched_pools, pools_848_matched], ignore_index=True)
#drop columns 23 - 38 and 43 -59 
all_matched_pools = all_matched_pools.drop(columns=all_matched_pools.columns[22:38].tolist() + all_matched_pools.columns[42:59].tolist())
all_matched_pools.info()

In [ ]:
#all_matched_pools.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/all_matched_pools.csv", index=False)

### now lets check how many of the admin pools are still missing from our matched dataset

In [ ]:
unmatched_baederleben_pools = baederleben_pools[~baederleben_pools["pool_id"].isin(all_matched_pools["pool_id"])]
unmatched_baederleben_pools.info()

In [ ]:
unmatched_baederleben_pools["address"] = unmatched_baederleben_pools.apply(
	lambda row: f"{row['street']}, {row['postalcode']} {row['city']}, Germany",
	axis=1
)

In [ ]:
## check for google data not included in the matched pools
unmatched_google_pools = google_pools[~google_pools["unique_identifier"].isin(all_matched_pools["unique_identifier"])]
#drop columns 15 to 30
unmatched_google_pools = unmatched_google_pools.drop(columns=unmatched_google_pools.columns[14:31].tolist())

unmatched_google_pools.info()

## lets try matching on address and then 500km radius

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from tqdm import tqdm
from fuzzywuzzy import fuzz
import re


class PoolMatcher:
    """
    Matches admin pools to Google Maps pools in two sequential steps:
    
    Step 1 — Spatial: for each admin pool find all gmaps pools within radius,
             keep only the CLOSEST one. Matched pairs are removed from the pool.
             
    Step 2 — Address: for remaining unmatched admin + gmaps pools, fuzzy-match
             on address string. Again keep only the BEST match per admin pool.
             Matched pairs are again removed from the pool.
             
    Result: every admin pool appears exactly once in the final dataframe,
            either as matched (spatial/address) or unmatched. Same for gmaps.
    """

    CONFIDENCE_THRESHOLDS = {
        'very_high': (0,   10),
        'high':      (10,  50),
        'medium':    (50,  100),
        'low':       (100, 500),
    }

    def __init__(self, radius_meters=500, address_similarity_threshold=0.7):
        self.radius_meters        = radius_meters
        self.sim_threshold        = address_similarity_threshold

        # populated during matching
        self.df_admin             = None   # cleaned, positional index
        self.df_gmaps             = None   # cleaned, positional index
        self.spatial_matches      = None   # DataFrame, one row per spatial match
        self.address_matches      = None   # DataFrame, one row per address match

    # ------------------------------------------------------------------
    # PUBLIC ENTRY POINT
    # ------------------------------------------------------------------

    def match(self, df_admin, df_gmaps,
              lat_col_admin='latitude', lon_col_admin='longitude',
              lat_col_gmaps='latitude', lon_col_gmaps='longitude',
              admin_address_cols=None, gmaps_address_cols=None):
        """
        Run full two-step matching pipeline.
        Returns a single tidy DataFrame (see get_results() for schema).
        """
        self._prepare_data(df_admin, df_gmaps,
                           lat_col_admin, lon_col_admin,
                           lat_col_gmaps, lon_col_gmaps)

        # --- Step 1: spatial ---
        print("\n=== STEP 1: SPATIAL MATCHING ===")
        unmatched_admin_idx, unmatched_gmaps_idx = self._spatial_match(
            lat_col_admin, lon_col_admin, lat_col_gmaps, lon_col_gmaps
        )
        self._print_spatial_summary()

        # --- Step 2: address (optional) ---
        if admin_address_cols and gmaps_address_cols:
            print("\n=== STEP 2: ADDRESS MATCHING ===")
            self._address_match(unmatched_admin_idx, unmatched_gmaps_idx,
                                admin_address_cols, gmaps_address_cols)
            self._print_address_summary(unmatched_admin_idx)
        else:
            self.address_matches = pd.DataFrame()
            print("\nSkipping address matching (no address columns provided)")

        # --- Build final dataframe ---
        print("\n=== BUILDING FINAL DATAFRAME ===")
        return self.get_results()

    # ------------------------------------------------------------------
    # STEP 0 — DATA PREPARATION
    # ------------------------------------------------------------------

    def _prepare_data(self, df_admin, df_gmaps,
                      lat_col_admin, lon_col_admin,
                      lat_col_gmaps, lon_col_gmaps):
        """Clean coordinates and reset to a clean positional index ONCE."""

        def clean(df, lat_col, lon_col, label):
            df = df.copy()
            df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
            df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
            before = len(df)
            df = df.dropna(subset=[lat_col, lon_col])
            # ↓ THE FIX: one authoritative positional reset right here
            df = df.reset_index(drop=True)
            print(f"{label}: {len(df):,} valid rows "
                  f"({before - len(df)} dropped for missing coordinates)")
            return df

        self.df_admin = clean(df_admin, lat_col_admin, lon_col_admin, "Admin")
        self.df_gmaps = clean(df_gmaps, lat_col_gmaps, lon_col_gmaps, "GMaps")

    # ------------------------------------------------------------------
    # STEP 1 — SPATIAL MATCHING
    # ------------------------------------------------------------------

    def _spatial_match(self, lat_col_admin, lon_col_admin,
                       lat_col_gmaps, lon_col_gmaps):
        """
        For every admin pool find all gmaps pools within radius.
        Keep only the CLOSEST gmaps pool per admin pool.
        Both sides of a matched pair are removed from the unmatched pool.
        Returns (unmatched_admin_indices, unmatched_gmaps_indices) as sets of
        positional integers.
        """
        coords_admin = np.radians(
            self.df_admin[[lat_col_admin, lon_col_admin]].values)
        coords_gmaps = np.radians(
            self.df_gmaps[[lat_col_gmaps, lon_col_gmaps]].values)

        tree        = BallTree(coords_gmaps, metric='haversine')
        radius_rad  = self.radius_meters / 6_371_000

        # query_radius returns all neighbours; we want distances too so we can
        # pick the closest one → use query with k=1 first, then radius filter
        # Actually query_radius with return_distance=True is cleanest:
        distances_list, indices_list = tree.query_radius(
            coords_admin, r=radius_rad, return_distance=True, sort_results=True
        )

        rows               = []
        matched_gmaps_pos  = set()   # gmaps positional indices already claimed

        for admin_pos in tqdm(range(len(self.df_admin)),
                              desc="Spatial matching"):
            neighbours     = indices_list[admin_pos]   # sorted nearest-first
            neighbour_dists = distances_list[admin_pos]

            # Walk through candidates nearest-first; take first unclaimed gmaps
            for gmaps_pos, dist_rad in zip(neighbours, neighbour_dists):
                if gmaps_pos in matched_gmaps_pos:
                    continue  # already claimed by a closer admin pool

                dist_m = dist_rad * 6_371_000
                rows.append({
                    'admin_pos':    admin_pos,
                    'gmaps_pos':    int(gmaps_pos),
                    'distance_m':   dist_m,
                    'confidence':   self._confidence(dist_m),
                    'match_type':   'spatial',
                })
                matched_gmaps_pos.add(int(gmaps_pos))
                break   # one best match per admin pool — done

        self.spatial_matches = pd.DataFrame(rows)

        matched_admin_pos  = set(self.spatial_matches['admin_pos'])
        unmatched_admin    = set(range(len(self.df_admin)))  - matched_admin_pos
        unmatched_gmaps    = set(range(len(self.df_gmaps)))  - matched_gmaps_pos

        return unmatched_admin, unmatched_gmaps

    # ------------------------------------------------------------------
    # STEP 2 — ADDRESS MATCHING
    # ------------------------------------------------------------------

    def _address_match(self, unmatched_admin_idx, unmatched_gmaps_idx,
                       admin_address_cols, gmaps_address_cols):
        """
        Fuzzy-match address strings for the pools that had no spatial match.
        Best match (highest similarity above threshold) wins.
        Each gmaps pool can be claimed at most once.
        """
        sub_admin = self.df_admin.loc[sorted(unmatched_admin_idx)].copy()
        sub_gmaps = self.df_gmaps.loc[sorted(unmatched_gmaps_idx)].copy()

        sub_admin['_addr'] = self._build_address_col(sub_admin, admin_address_cols)
        sub_gmaps['_addr'] = self._build_address_col(sub_gmaps, gmaps_address_cols)

        # drop rows with no usable address
        sub_admin = sub_admin[sub_admin['_addr'].str.len() >= 5]
        sub_gmaps = sub_gmaps[sub_gmaps['_addr'].str.len() >= 5]

        print(f"Address matching: {len(sub_admin):,} admin vs "
              f"{len(sub_gmaps):,} gmaps pools")

        admin_addrs = sub_admin['_addr'].tolist()
        gmaps_addrs = sub_gmaps['_addr'].tolist()
        admin_pos_list = sub_admin.index.tolist()   # positional (already reset)
        gmaps_pos_list = sub_gmaps.index.tolist()

        gmaps_cleaned = [self._clean_addr(a) for a in gmaps_addrs]
        claimed_gmaps = set()
        rows = []

        for i, (admin_pos, raw_admin) in enumerate(
                zip(admin_pos_list, admin_addrs)):

            clean_admin = self._clean_addr(raw_admin)
            if len(clean_admin) < 5:
                continue

            best_score   = self.sim_threshold * 100   # must beat threshold
            best_gmaps_pos  = None
            best_gmaps_addr = None

            for j, (gmaps_pos, raw_gmaps, clean_gmaps) in enumerate(
                    zip(gmaps_pos_list, gmaps_addrs, gmaps_cleaned)):

                if gmaps_pos in claimed_gmaps:
                    continue
                if len(clean_gmaps) < 5:
                    continue

                score = fuzz.token_sort_ratio(clean_admin, clean_gmaps)
                if score > best_score:
                    best_score      = score
                    best_gmaps_pos  = gmaps_pos
                    best_gmaps_addr = raw_gmaps

            if best_gmaps_pos is not None:
                rows.append({
                    'admin_pos':      admin_pos,
                    'gmaps_pos':      best_gmaps_pos,
                    'admin_address':  raw_admin,
                    'gmaps_address':  best_gmaps_addr,
                    'similarity':     best_score / 100,
                    'match_type':     'address',
                    'confidence':     'address',
                    'distance_m':     None,   # filled in below
                })
                claimed_gmaps.add(best_gmaps_pos)

        self.address_matches = pd.DataFrame(rows)

        if len(self.address_matches) > 0:
            self._fill_address_distances()

    def _fill_address_distances(self):
        """Calculate haversine distances for address-matched pairs."""
        dists = []
        for _, row in self.address_matches.iterrows():
            a = self.df_admin.loc[row['admin_pos']]
            g = self.df_gmaps.loc[row['gmaps_pos']]
            dists.append(self._haversine(
                a['latitude'], a['longitude'],
                g['latitude'], g['longitude']
            ))
        self.address_matches['distance_m'] = dists

    # ------------------------------------------------------------------
    # RESULTS
    # ------------------------------------------------------------------

    def get_results(self):
        """
        Build and return a single tidy DataFrame.

        Columns always present:
            match_type      : spatial_very_high / spatial_high / spatial_medium /
                              spatial_low / address / unmatched_admin / unmatched_gmaps
            admin_pos       : positional index into df_admin  (NaN for unmatched_gmaps)
            gmaps_pos       : positional index into df_gmaps  (NaN for unmatched_admin)
            distance_m      : metres between matched pair     (NaN if address or unmatched)
            confidence      : very_high/high/medium/low/address/unmatched
            similarity      : fuzzy score 0-1                 (address matches only)
            
        Followed by all admin columns (suffixed _admin on clash) and all gmaps
        columns (suffixed _gmaps on clash).
        """
        parts = []

        # ---- spatial matches ----
        if self.spatial_matches is not None and len(self.spatial_matches) > 0:
            sp = self.spatial_matches.copy()
            sp['match_type'] = 'spatial_' + sp['confidence']
            sp = sp.merge(self.df_admin, left_on='admin_pos',
                          right_index=True, suffixes=('', '_admin'))
            sp = sp.merge(self.df_gmaps, left_on='gmaps_pos',
                          right_index=True, suffixes=('_admin', '_gmaps'))
            parts.append(sp)

        # ---- address matches ----
        if self.address_matches is not None and len(self.address_matches) > 0:
            am = self.address_matches.copy()
            am = am.merge(self.df_admin, left_on='admin_pos',
                          right_index=True, suffixes=('', '_admin'))
            am = am.merge(self.df_gmaps, left_on='gmaps_pos',
                          right_index=True, suffixes=('_admin', '_gmaps'))
            parts.append(am)

        # ---- determine which indices are matched ----
        matched_admin = set()
        matched_gmaps = set()
        if self.spatial_matches is not None and len(self.spatial_matches):
            matched_admin |= set(self.spatial_matches['admin_pos'])
            matched_gmaps |= set(self.spatial_matches['gmaps_pos'])
        if self.address_matches is not None and len(self.address_matches):
            matched_admin |= set(self.address_matches['admin_pos'])
            matched_gmaps |= set(self.address_matches['gmaps_pos'])

        # ---- unmatched admin ----
        ua = self.df_admin[~self.df_admin.index.isin(matched_admin)].copy()
        ua['match_type'] = 'unmatched_admin'
        ua['admin_pos']  = ua.index
        ua['confidence'] = 'unmatched'
        parts.append(ua)

        # ---- unmatched gmaps ----
        ug = self.df_gmaps[~self.df_gmaps.index.isin(matched_gmaps)].copy()
        ug['match_type'] = 'unmatched_gmaps'
        ug['gmaps_pos']  = ug.index
        ug['confidence'] = 'unmatched'
        parts.append(ug)

        final = pd.concat(parts, ignore_index=True)

        # ---- tidy column order ----
        lead = ['match_type', 'admin_pos', 'gmaps_pos',
                'distance_m', 'confidence', 'similarity']
        lead = [c for c in lead if c in final.columns]
        rest = [c for c in final.columns if c not in lead]
        final = final[lead + rest]

        # ---- sort ----
        order = {'spatial_very_high': 0, 'spatial_high': 1,
                 'spatial_medium': 2,    'spatial_low': 3,
                 'address': 4, 'unmatched_admin': 5, 'unmatched_gmaps': 6}
        final['_o'] = final['match_type'].map(order)
        final = (final.sort_values(['_o', 'admin_pos', 'distance_m'])
                      .drop(columns='_o')
                      .reset_index(drop=True))

        self._print_final_summary(final, matched_admin, matched_gmaps)
        return final

    # ------------------------------------------------------------------
    # HELPERS
    # ------------------------------------------------------------------

    @staticmethod
    def _haversine(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        a = (np.sin((lat2-lat1)/2)**2
             + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2)
        return 2 * np.arcsin(np.sqrt(a)) * 6_371_000

    @staticmethod
    def _confidence(dist_m):
        if dist_m <= 10:  return 'very_high'
        if dist_m <= 50:  return 'high'
        if dist_m <= 100: return 'medium'
        return 'low'

    @staticmethod
    def _build_address_col(df, cols):
        def row_addr(row):
            return ' '.join(
                str(row[c]).strip()
                for c in cols
                if c in row and pd.notna(row[c]) and str(row[c]).strip()
            ).lower()
        return df.apply(row_addr, axis=1)

    @staticmethod
    def _clean_addr(addr):
        addr = re.sub(r'[^\w\s]', ' ', addr)
        addr = re.sub(r'\s+', ' ', addr)
        for full, short in [('street','st'),('avenue','ave'),('road','rd'),
                             ('boulevard','blvd'),('drive','dr'),('lane','ln')]:
            addr = re.sub(r'\b'+full+r'\b', short, addr)
        return addr.strip()

    # ------------------------------------------------------------------
    # PRINT HELPERS
    # ------------------------------------------------------------------

    def _print_spatial_summary(self):
        n_admin   = len(self.df_admin)
        n_matched = len(self.spatial_matches)
        print(f"\n--- Spatial results ---")
        print(f"  Admin pools:    {n_admin:,}")
        print(f"  Matched:        {n_matched:,}  ({n_matched/n_admin*100:.1f}%)")
        print(f"  Unmatched:      {n_admin - n_matched:,}")
        if n_matched:
            for level in ['very_high','high','medium','low']:
                c = (self.spatial_matches['confidence'] == level).sum()
                label = {'very_high':'0-10m','high':'10-50m',
                         'medium':'50-100m','low':'100-500m'}[level]
                print(f"    {label:<10}: {c:,}")

    def _print_address_summary(self, unmatched_admin_before):
        n = len(self.address_matches) if self.address_matches is not None else 0
        print(f"\n--- Address results ---")
        print(f"  Pools entering address step: {len(unmatched_admin_before):,}")
        print(f"  Address matches found:       {n:,}")

    def _print_final_summary(self, final, matched_admin, matched_gmaps):
        n_admin = len(self.df_admin)
        n_gmaps = len(self.df_gmaps)
        total_matched = len(matched_admin)
        print(f"\n=== FINAL SUMMARY ===")
        print(f"  Total admin pools:           {n_admin:,}")
        print(f"  Total gmaps pools:           {n_gmaps:,}")
        print(f"  Matched pairs:               {total_matched:,}")
        print(f"  Unmatched admin pools:       {n_admin - total_matched:,}")
        print(f"  Unmatched gmaps pools:       {n_gmaps - len(matched_gmaps):,}")
        print(f"\n  Rows by match_type:")
        for mt, cnt in final['match_type'].value_counts().sort_index().items():
            print(f"    {mt:<25}: {cnt:,}")
        print(f"  Total rows: {len(final):,}")

In [ ]:
def run_complete_matching_analysis(df_admin, df_gmaps,
                                   admin_address_cols=None,
                                   gmaps_address_cols=None,
                                   radius_meters=500,
                                   similarity_threshold=0.7,
                                   normalize_dtypes_fn=None):
    """
    Two-step pool matcher: spatial first, then address for leftovers.
    
    Returns
    -------
    DataFrame with one row per admin pool (matched or unmatched),
    plus rows for unmatched gmaps pools at the bottom.
    """
    print("=== POOL MATCHING ANALYSIS ===")
    print(f"Admin pools : {len(df_admin):,}")
    print(f"GMaps pools : {len(df_gmaps):,}")

    # Optionally normalise dtypes before matching
    if normalize_dtypes_fn:
        df_admin = normalize_dtypes_fn(df_admin.copy())
        df_gmaps = normalize_dtypes_fn(df_gmaps.copy())

    matcher = PoolMatcher(
        radius_meters=radius_meters,
        address_similarity_threshold=similarity_threshold,
    )

    results = matcher.match(
        df_admin, df_gmaps,
        admin_address_cols=admin_address_cols,
        gmaps_address_cols=gmaps_address_cols,
    )

    return results, matcher   # return matcher so caller can inspect internals

In [ ]:
# Option 2: Complete matching with addresses (returns analyzer + combined_stats)
matched_df, stats = run_complete_matching_analysis(
    unmatched_baederleben_pools, unmatched_google_pools,
    admin_address_cols=['address'],
    gmaps_address_cols=['address']
)


In [ ]:
bd_gmaps =matched_df[matched_df["match_type"].isin([ "spatial_high", "spatial_medium", "spatial_low","address"])].dropna(axis=1, how="all")
# check for duplicates in the matched dataset
duplicates = bd_gmaps[bd_gmaps.duplicated(subset=['admin_pos'], keep=False)]
print(f"Number of duplicate admin_pos entries in matched dataset: {len(duplicates)}")
print("Duplicate admin_pos entries:")
print(duplicates[['admin_pos', 'gmaps_pos', 'match_type']])

In [ ]:
bd_gmaps.info()

In [ ]:
#remove bd_gmaps from unmatched_google_pools and unmatched_baederleben_pools
matched_admin_pos = set(bd_gmaps['pool_id'])
matched_gmaps_pos = set(bd_gmaps['place_id'])
unmatched_baederleben_pools_final = unmatched_baederleben_pools[~unmatched_baederleben_pools['pool_id'].isin(matched_admin_pos)]
unmatched_google_pools_final = unmatched_google_pools[~unmatched_google_pools['place_id'].isin(matched_gmaps_pos)]

In [ ]:
unmatched_baederleben_pools_final.info()

In [ ]:
unmatched_google_pools_final.info()

### lets merge all the matched baederleben pools and the gmaps poos that did not match any baederleben pools

In [ ]:
cols = ['pool_id', 'pool_name', 'place_id', 'unique_identifier']

df_joined = pd.concat([
    all_matched_pools[cols],
    bd_gmaps[cols],
    unmatched_google_pools_final.reindex(columns=cols)  # fills missing columns with NaN
], ignore_index=True)
df_joined.info()

### After checking all the pools, I manually searched for the google matches for the 336 that did not match any google pools.
### Below I am merging the place id and unique identifiers for all possible swimming pools. This includes all the matches for the baederleben pools found on google, the ones that matched our existing google data and those that are only in or google data. 
### we have collected data for some of these but for  clarity we would collect all the place jsons again then for the reviews we would only update those we have and collect for those that we do not have.
### Of course some places even though in the baederleben list are not of interest to us. eg. swimming schools etc

In [ ]:
#now let's add the ones that we manually matched to the all_pools dataset
manual_matches = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/manually_matched_pools.csv")
manual_matches.info()

In [ ]:
#concatenate the manually matched pools with the all_pools dataset
all_pools_updated = pd.concat([df_joined[["pool_id","pool_name","place_id","unique_identifier"]], manual_matches[["pool_id","pool_name","unique_identifier"]]], ignore_index=True)
all_pools_updated.info()

In [ ]:
#how many baederleben pool ids are in all pools
bd_missed = baederleben_pools[~baederleben_pools['pool_id'].isin(all_pools_updated['pool_id'])]
bd_missed['address'] = bd_missed.apply(lambda row: f"{row['street']}, {row['postalcode']} {row['city']}, Germany", axis=1)
# bd_missed.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/baederleben_pools_missed.csv", index=False)
bd_missed.info()

In [ ]:
## now we need to find the missing uids for the all_possible_pools dataset
uid_map = (
    google_pools.dropna(subset=["place_id", "unique_identifier"])
    .drop_duplicates(subset=["place_id"])
    .set_index("place_id")["unique_identifier"]
)

missing_uid_mask = all_pools_updated["unique_identifier"].isna()
all_pools_updated.loc[missing_uid_mask, "unique_identifier"] = (
    all_pools_updated.loc[missing_uid_mask, "place_id"].map(uid_map)
)

all_pools_updated.info()

In [ ]:
# change the data type for pool_id to  int64
all_pools_updated["pool_id"] = pd.to_numeric(all_pools_updated["pool_id"], errors="coerce").astype("Int64")
all_pools_updated.info()

In [ ]:
#check for duplicate pool_id and pool name in the all_pools_updated dataset where pool id is not null 
duplicate_pool_id = all_pools_updated[all_pools_updated["pool_id"].duplicated(keep=False) & all_pools_updated["pool_id"].notna()]
print(f"Number of duplicate pool_id entries: {len(duplicate_pool_id)}")
print("Duplicate pool_id entries:")
display(duplicate_pool_id)

In [ ]:
# Split into rows with and without pool_id
has_pool_id = all_pools_updated[all_pools_updated['pool_id'].notna()]
no_pool_id = all_pools_updated[all_pools_updated['pool_id'].isna()]

# Deduplicate only rows that have a pool_id
has_pool_id = has_pool_id.sort_values(by=["unique_identifier", "place_id"], key=lambda col: col.notna(), ascending=False)
has_pool_id = has_pool_id.drop_duplicates(subset=["pool_id"], keep="first")

# Recombine
all_pools_updated = pd.concat([has_pool_id, no_pool_id]).reset_index(drop=True)
all_pools_updated.info()

In [ ]:
all_pools_updated.shape

### Now we save the `all_pools_updated` file and run the places info on it  again

In [ ]:
#all_pools_updated.to_csv("/home/ethel/Desktop/swimming_pool_project/icwsm_may_02042026/data/pool_selection/all_inclusive_possible_pools_to_use_in_project_09042026.csv", index=False)